In [1]:
import pandas as pd

In [2]:
genome_tags = pd.read_csv("../datasets/genome-tags.csv")
genome_tags.head()

,tagId,tag
0,1,007
1,2,007 (series)
2,3,18th century
3,4,1920s
4,5,1930s


In [3]:
genome_tags.shape

(1128, 2)

In [4]:
tags = pd.read_csv("../datasets/tags.csv")
tags.head()

,userId,movieId,tag,timestamp
0,10,260,good vs evil,1430666558
1,10,260,Harrison Ford,1430666505
2,10,260,sci-fi,1430666538
3,14,1221,Al Pacino,1311600756
4,14,1221,mafia,1311600746


In [5]:
tags.shape

(2328315, 4)

In [6]:
genre = pd.read_csv("../datasets/movies.csv")
genre.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [7]:
genre.shape

(86537, 3)

## Milestone 1: Dataset audit and clean catalog

Confirm the MovieLens release, inspect data quality and movie-ID joins across movies, user tags, and Genome data, then build a movie-level catalog with feature coverage. Genome scores are streamed in chunks because the file is large.

In [8]:
import numpy as np
from IPython.display import display

# The local README identifies this as ml-latest, generated on July 20, 2023.
movie_ids = set(genre["movieId"].dropna().astype("int64"))
genome_tag_ids = set(genome_tags["tagId"].dropna().astype("int64"))

missing_movie_ids = int(genre["movieId"].isna().sum())
duplicate_movie_ids = int(genre["movieId"].duplicated().sum())
if missing_movie_ids or duplicate_movie_ids:
    raise ValueError(
        "movies.csv must have a non-null, unique movieId for a reliable catalog join. "
        f"Missing IDs: {missing_movie_ids}; duplicate IDs: {duplicate_movie_ids}."
    )

tag_movie_ids = set(tags["movieId"].dropna().astype("int64"))
clean_tags = tags.dropna(subset=["movieId", "tag"]).copy()
clean_tags["tag"] = clean_tags["tag"].astype("string").str.strip()
blank_tag_rows = int(clean_tags["tag"].eq("").sum())
clean_tags = clean_tags.loc[
    clean_tags["tag"].ne("") & clean_tags["movieId"].isin(movie_ids)
].copy()
tag_counts = clean_tags.groupby("movieId").size().rename("user_tag_count")

### Genome score audit

Scan the dense movie-by-tag scores in chunks, checking score ranges, movie/tag IDs, duplicate pairs, and per-movie completeness.

In [9]:
score_rows = 0
score_movie_counts = pd.Series(dtype="int64")
score_movie_ids = set()
score_tag_ids = set()
null_score_movie_ids = 0
null_score_tag_ids = 0
null_relevance_scores = 0
invalid_relevance_scores = 0
duplicate_score_pairs = 0
out_of_order_score_chunks = 0
previous_score_pair = None

for score_chunk in pd.read_csv(
    "../datasets/genome-scores.csv",
    usecols=["movieId", "tagId", "relevance"],
    chunksize=1_000_000,
):
    score_rows += len(score_chunk)
    null_score_movie_ids += int(score_chunk["movieId"].isna().sum())
    null_score_tag_ids += int(score_chunk["tagId"].isna().sum())

    relevance = pd.to_numeric(score_chunk["relevance"], errors="coerce")
    null_relevance_scores += int(relevance.isna().sum())
    invalid_relevance_scores += int((~relevance.between(0, 1) & relevance.notna()).sum())

    valid_pairs = score_chunk.dropna(subset=["movieId", "tagId"])[["movieId", "tagId"]]
    valid_pairs = valid_pairs.astype("int64")
    pair_index = pd.MultiIndex.from_frame(valid_pairs)
    duplicate_score_pairs += int(pair_index.duplicated().sum())
    if not pair_index.is_monotonic_increasing:
        out_of_order_score_chunks += 1
    if len(pair_index):
        first_pair = tuple(pair_index[0])
        if previous_score_pair == first_pair:
            duplicate_score_pairs += 1
        previous_score_pair = tuple(pair_index[-1])

    chunk_movie_counts = score_chunk.groupby("movieId").size()
    score_movie_counts = score_movie_counts.add(chunk_movie_counts, fill_value=0)
    score_movie_ids.update(score_chunk["movieId"].dropna().astype("int64").unique())
    score_tag_ids.update(score_chunk["tagId"].dropna().astype("int64").unique())

score_movie_counts = score_movie_counts.astype("int64")
unknown_tag_movie_ids = set(tags["movieId"].dropna().astype("int64")) - movie_ids
unknown_score_movie_ids = score_movie_ids - movie_ids
unknown_score_tag_ids = score_tag_ids - genome_tag_ids
expected_scores_per_movie = len(genome_tag_ids)
movies_with_incomplete_genome = int(
    (score_movie_counts != expected_scores_per_movie).sum()
)

### Assemble catalog and review findings

Join feature-coverage counts onto the complete movie list, keeping movies with missing tag sources so they can be handled explicitly in later milestones.

In [10]:
movie_catalog = genre.copy()
movie_catalog = movie_catalog.merge(
    tag_counts, left_on="movieId", right_index=True, how="left"
)
movie_catalog = movie_catalog.merge(
    score_movie_counts.rename("genome_score_count"),
    left_on="movieId",
    right_index=True,
    how="left",
)
movie_catalog[["user_tag_count", "genome_score_count"]] = (
    movie_catalog[["user_tag_count", "genome_score_count"]]
    .fillna(0)
    .astype("int64")
)
movie_catalog["has_user_tags"] = movie_catalog["user_tag_count"].gt(0)
movie_catalog["has_genome_scores"] = movie_catalog["genome_score_count"].gt(0)

audit_report = pd.DataFrame(
    [
        ("Dataset release", "ml-latest; generated 2023-07-20 (datasets/README.txt)"),
        ("Movie rows", len(genre)),
        ("Unique movie IDs", genre["movieId"].nunique()),
        ("Movies missing title", int(genre["title"].isna().sum())),
        ("Movies missing genres", int(genre["genres"].isna().sum())),
        ("User-tag rows", len(tags)),
        ("Distinct cleaned user-tag strings", clean_tags["tag"].nunique(dropna=True)),
        ("Clean user-tag rows", len(clean_tags)),
        ("Movies with user tags", clean_tags["movieId"].nunique()),
        ("User-tag rows with unknown movie IDs", len(unknown_tag_movie_ids)),
        ("User-tag rows with missing movieId", int(tags["movieId"].isna().sum())),
        ("User-tag rows with missing tag text", int(tags["tag"].isna().sum())),
        ("User-tag rows with blank tag text", blank_tag_rows),
        ("Genome tag rows", len(genome_tags)),
        ("Unique Genome tag IDs", len(genome_tag_ids)),
        ("Genome score rows", score_rows),
        ("Movies with Genome scores", len(score_movie_ids & movie_ids)),
        ("Genome score movie IDs absent from movies.csv", len(unknown_score_movie_ids)),
        ("Genome score tag IDs absent from genome-tags.csv", len(unknown_score_tag_ids)),
        ("Genome scores with missing movieId", null_score_movie_ids),
        ("Genome scores with missing tagId", null_score_tag_ids),
        ("Genome scores with missing relevance", null_relevance_scores),
        ("Genome relevance values outside [0, 1]", invalid_relevance_scores),
        ("Duplicate movie-tag pairs detected", duplicate_score_pairs),
        ("Genome score chunks out of movie/tag order", out_of_order_score_chunks),
        ("Movies with a non-dense Genome score row count", movies_with_incomplete_genome),
        ("Movies with no user tags", int((~movie_catalog["has_user_tags"]).sum())),
        ("Movies with no Genome scores", int((~movie_catalog["has_genome_scores"]).sum())),
    ],
    columns=["Audit check", "Result"],
)

display(audit_report)
display(movie_catalog.head())

,Audit check,Result
0,Dataset release,ml-latest; generated 2023-07-20 (datasets/READ...
1,Movie rows,86537
2,Unique movie IDs,86537
3,Movies missing title,0
4,Movies missing genres,0
5,User-tag rows,2328315
6,Distinct cleaned user-tag strings,153476
7,Clean user-tag rows,2328288
8,Movies with user tags,53452
9,User-tag rows with unknown movie IDs,0


,movieId,title,genres,user_tag_count,genome_score_count,has_user_tags,has_genome_scores
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1440,1128,True,True
1,2,Jumanji (1995),Adventure|Children|Fantasy,653,1128,True,True
2,3,Grumpier Old Men (1995),Comedy|Romance,36,1128,True,True
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,13,1128,True,True
4,5,Father of the Bride Part II (1995),Comedy,68,1128,True,True


## Milestone 2: Create the feature representations

Turn each movie's genres, cleaned user-generated tags, and Genome relevance scores into separate, row-aligned sparse matrices. Normalize the feature groups independently and inspect their coverage before combining or comparing movies.

### Genre features

Encode each movie's genres as a multi-hot vector, then L2-normalize rows so movies with more genre labels do not automatically have larger vectors.

In [11]:
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.preprocessing import MultiLabelBinarizer, normalize

catalog_movie_ids = movie_catalog["movieId"].to_numpy(dtype="int64")
genre_labels = genre.set_index("movieId").loc[catalog_movie_ids, "genres"]
genre_lists = genre_labels.str.split("|", regex=False).tolist()

genre_encoder = MultiLabelBinarizer(sparse_output=True)
genre_matrix = genre_encoder.fit_transform(genre_lists).astype(np.float32).tocsr()
genre_matrix = normalize(genre_matrix, norm="l2", copy=False).tocsr()
genre_feature_names = genre_encoder.classes_

print(f"Genre matrix: {genre_matrix.shape[0]:,} movies x {genre_matrix.shape[1]} genres")

Genre matrix: 86,537 movies x 20 genres


### User-generated tag features

Normalize tag casing and spacing, deduplicate repeated tag assignments within each movie, and use TF-IDF to reduce the influence of tags common across many movies.

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

normalized_tags = clean_tags[["movieId", "tag"]].copy()
normalized_tags["tag"] = (
    normalized_tags["tag"]
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
normalized_tags = normalized_tags.drop_duplicates(["movieId", "tag"])
tag_documents = normalized_tags.assign(
    tag_token=normalized_tags["tag"].str.replace(" ", "_", regex=False)
).groupby("movieId")["tag_token"].agg(" ".join)
tag_documents = tag_documents.reindex(catalog_movie_ids, fill_value="")

tag_vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    token_pattern=None,
    lowercase=False,
    dtype=np.float32,
)
user_tag_matrix = tag_vectorizer.fit_transform(tag_documents).tocsr()
user_tag_matrix = normalize(user_tag_matrix, norm="l2", copy=False).tocsr()
user_tag_feature_names = tag_vectorizer.get_feature_names_out()

print(
    f"User-tag matrix: {user_tag_matrix.shape[0]:,} movies x "
    f"{user_tag_matrix.shape[1]:,} tag features"
)

User-tag matrix: 86,537 movies x 143,212 tag features


### Genome relevance features

Map each movie/tag pair to a row/column in a sparse matrix, preserving relevance scores as continuous values. Normalize each movie row; movies outside the Genome set remain zero rows.

In [13]:
genome_tag_ids_ordered = sorted(genome_tag_ids)
catalog_movie_row = {movie_id: row for row, movie_id in enumerate(catalog_movie_ids)}
genome_tag_column = {tag_id: column for column, tag_id in enumerate(genome_tag_ids_ordered)}

genome_rows = np.empty(score_rows, dtype=np.int32)
genome_columns = np.empty(score_rows, dtype=np.int32)
genome_values = np.empty(score_rows, dtype=np.float32)
offset = 0

for score_chunk in pd.read_csv(
    "../datasets/genome-scores.csv",
    usecols=["movieId", "tagId", "relevance"],
    chunksize=1_000_000,
):
    chunk_size = len(score_chunk)
    next_offset = offset + chunk_size
    genome_rows[offset:next_offset] = score_chunk["movieId"].map(catalog_movie_row).to_numpy(dtype=np.int32)
    genome_columns[offset:next_offset] = score_chunk["tagId"].map(genome_tag_column).to_numpy(dtype=np.int32)
    genome_values[offset:next_offset] = score_chunk["relevance"].to_numpy(dtype=np.float32)
    offset = next_offset

if offset != score_rows:
    raise ValueError(f"Expected {score_rows:,} Genome scores, read {offset:,}.")

genome_matrix = csr_matrix(
    (genome_values, (genome_rows, genome_columns)),
    shape=(len(catalog_movie_ids), len(genome_tag_ids_ordered)),
    dtype=np.float32,
)
genome_matrix = normalize(genome_matrix, norm="l2", copy=False).tocsr()
genome_feature_names = (
    genome_tags.set_index("tagId").loc[genome_tag_ids_ordered, "tag"].to_numpy()
)

print(
    f"Genome matrix: {genome_matrix.shape[0]:,} movies x "
    f"{genome_matrix.shape[1]:,} Genome tags"
)

Genome matrix: 86,537 movies x 1,128 Genome tags


### Check feature coverage and examples

Confirm that all matrices use the same movie order, compare the fraction of movies with nonzero rows, and inspect the strongest features for one familiar title.

In [14]:
feature_matrices = {
    "genres": genre_matrix,
    "user tags (TF-IDF)": user_tag_matrix,
    "Genome relevance": genome_matrix,
}
assert all(matrix.shape[0] == len(movie_catalog) for matrix in feature_matrices.values())

feature_coverage = pd.DataFrame(
    [
        {
            "Feature group": name,
            "Movies": matrix.shape[0],
            "Features": matrix.shape[1],
            "Movies with features": int(np.count_nonzero(matrix.getnnz(axis=1))),
            "Nonzero values": matrix.nnz,
            "Storage": "sparse",
        }
        for name, matrix in feature_matrices.items()
    ]
)
display(feature_coverage)

example_title = "Toy Story (1995)"
example_position = movie_catalog.index[movie_catalog["title"].eq(example_title)][0]

def strongest_features(matrix, feature_names, row, limit=12):
    values = matrix.getrow(row).toarray().ravel()
    top_indices = np.argsort(values)[-limit:][::-1]
    return pd.DataFrame(
        {
            "Feature": np.asarray(feature_names)[top_indices],
            "Weight": values[top_indices],
        }
    ).query("Weight > 0")

print(f"Strongest features for {example_title}")
for name, matrix, feature_names in [
    ("Genres", genre_matrix, genre_feature_names),
    ("User tags", user_tag_matrix, user_tag_feature_names),
    ("Genome tags", genome_matrix, genome_feature_names),
]:
    print(name)
    display(strongest_features(matrix, feature_names, example_position))

,Feature group,Movies,Features,Movies with features,Nonzero values,Storage
0,genres,86537,20,86537,152288,sparse
1,user tags (TF-IDF),86537,143212,53452,1142562,sparse
2,Genome relevance,86537,1128,16376,18472128,sparse


Strongest features for Toy Story (1995)
Genres


,Feature,Weight
0,Fantasy,0.447214
1,Adventure,0.447214
2,Animation,0.447214
3,Children,0.447214
4,Comedy,0.447214


User tags


,Feature,Weight
0,forced_puns,0.060645
1,turbo_boost,0.060645
2,https://movielens.org/tag/:animation,0.060645
3,toys_played,0.060645
4,pizza_van,0.060645
5,reference_to_mattel,0.060645
6,ad_for_toys,0.060645
7,first_cgi_film,0.060645
8,sleeping_dog,0.060645
9,mixed_soundtrack,0.060645


Genome tags


,Feature,Weight
0,toys,0.112750
1,computer animation,0.112666
2,pixar animation,0.112045
3,animation,0.111256
4,kids and family,0.111143
5,kids,0.110438
6,pixar,0.108718
7,children,0.107928
8,animated,0.107843
9,cartoon,0.106744


## Milestone 3: Combine features and retrieve similar movies

Combine the normalized feature groups with tunable weights, then find nearest movies using cosine distance. The example output compares the combined recommender with each feature group on its own so we can inspect what each source contributes.

### Set feature weights and build the combined representation

Start with equal influence for genres, user tags, and Genome tags. Since each feature block is already L2-normalized, multiply each block by the square root of its weight; cosine similarity then reflects the chosen block contributions. Normalize each combined movie row afterward.

In [15]:
from scipy.sparse import hstack

feature_weights = {
    "genres": 1 / 3,
    "user_tags": 1 / 3,
    "genome": 1 / 3,
}

if not np.isclose(sum(feature_weights.values()), 1.0):
    raise ValueError("Feature weights must sum to 1.")
if any(weight < 0 for weight in feature_weights.values()):
    raise ValueError("Feature weights must be non-negative.")

weighted_feature_blocks = [
    genre_matrix.multiply(np.sqrt(feature_weights["genres"])),
    user_tag_matrix.multiply(np.sqrt(feature_weights["user_tags"])),
    genome_matrix.multiply(np.sqrt(feature_weights["genome"])),
]
combined_matrix = hstack(weighted_feature_blocks, format="csr", dtype=np.float32)
combined_matrix = normalize(combined_matrix, norm="l2", axis=1, copy=False).tocsr()

print(f"Feature weights: {feature_weights}")
print(f"Combined matrix: {combined_matrix.shape[0]:,} movies x {combined_matrix.shape[1]:,} features")
print(f"Combined matrix nonzero values: {combined_matrix.nnz:,}")

Feature weights: {'genres': 0.3333333333333333, 'user_tags': 0.3333333333333333, 'genome': 0.3333333333333333}
Combined matrix: 86,537 movies x 144,360 features
Combined matrix nonzero values: 19,766,978


### Resolve flexible movie-title input

Prepare normalized title aliases so people can search without the release year or MovieLens's inverted article format (for example, `The Shawshank Redemption`). Exact matches are preferred; when multiple films share a title, use the version with the most user-tag data and show which year was selected.

In [16]:
import difflib
import re

ARTICLE_PATTERN = re.compile(r"^(.*),\s*(the|a|an)$", flags=re.IGNORECASE)
YEAR_PATTERN = re.compile(r"\s*\((\d{4})\)\s*$")

def normalize_title_text(value):
    """Normalize a title for matching, ignoring punctuation and case."""
    without_year = YEAR_PATTERN.sub("", str(value)).strip()
    return re.sub(r"[^a-z0-9]+", " ", without_year.casefold()).strip()

def title_aliases(value):
    """Return normalized aliases, including MovieLens's trailing-article form."""
    title_without_year = YEAR_PATTERN.sub("", str(value)).strip()
    aliases = {normalize_title_text(title_without_year)}
    article_match = ARTICLE_PATTERN.match(title_without_year)
    if article_match:
        aliases.add(normalize_title_text(f"{article_match.group(2)} {article_match.group(1)}"))
    else:
        first_word, separator, remaining = title_without_year.partition(" ")
        if separator and first_word.casefold() in {"the", "a", "an"}:
            aliases.add(normalize_title_text(f"{remaining}, {first_word}"))
    return {alias for alias in aliases if alias}

title_alias_to_positions = {}
for position, stored_title in enumerate(movie_catalog["title"]):
    for alias in title_aliases(stored_title):
        title_alias_to_positions.setdefault(alias, []).append(position)

def display_movie_title(position):
    """Show a user-friendly title while preserving its year for disambiguation."""
    stored_title = movie_catalog.iloc[position]["title"]
    article_match = ARTICLE_PATTERN.match(YEAR_PATTERN.sub("", stored_title).strip())
    if article_match:
        title_without_year = f"{article_match.group(2)} {article_match.group(1)}"
    else:
        title_without_year = YEAR_PATTERN.sub("", stored_title).strip()
    year_match = YEAR_PATTERN.search(stored_title)
    return f"{title_without_year} ({year_match.group(1)})" if year_match else title_without_year

def resolve_movie_title(query):
    """Resolve a natural title query to one movie position or explain ambiguity."""
    query = str(query).strip()
    if not query:
        raise ValueError("Enter a movie title to search.")

    year_match = YEAR_PATTERN.search(query)
    requested_year = year_match.group(1) if year_match else None
    query_aliases = title_aliases(query)
    exact_positions = sorted(
        {position for alias in query_aliases for position in title_alias_to_positions.get(alias, [])}
    )

    if requested_year and exact_positions:
        title_positions = exact_positions.copy()
        exact_positions = [
            position
            for position in exact_positions
            if (match := YEAR_PATTERN.search(movie_catalog.iloc[position]["title"]))
            and match.group(1) == requested_year
        ]
        if not exact_positions:
            available_years = sorted(
                {
                    match.group(1)
                    for position in title_positions
                    if (match := YEAR_PATTERN.search(movie_catalog.iloc[position]["title"]))
                }
            )
            raise ValueError(
                f"No movie matched {query!r}. Available year(s) for this title: "
                f"{', '.join(available_years) or 'not listed'}"
            )
    if len(exact_positions) == 1:
        return exact_positions[0]
    if len(exact_positions) > 1:
        if not requested_year:
            selected_position = max(
                exact_positions,
                key=lambda position: (
                    movie_catalog.iloc[position]["user_tag_count"],
                    movie_catalog.iloc[position]["genome_score_count"],
                    YEAR_PATTERN.search(movie_catalog.iloc[position]["title"]).group(1)
                    if YEAR_PATTERN.search(movie_catalog.iloc[position]["title"])
                    else "0000",
                ),
            )
            print(
                f"Several movies match {query!r}; using {display_movie_title(selected_position)} "
                "(the match with the most user-tag data). Add a year to choose another version."
            )
            return selected_position
        choices = ", ".join(display_movie_title(position) for position in exact_positions[:8])
        raise ValueError(f"More than one movie matches {query!r}. Please specify the year: {choices}")

    normalized_query = normalize_title_text(query)
    if len(normalized_query) >= 3:
        partial_positions = sorted(
            {
                position
                for alias, positions in title_alias_to_positions.items()
                if normalized_query in alias
                for position in positions
            },
            key=lambda position: display_movie_title(position).casefold(),
        )
        if len(partial_positions) == 1:
            return partial_positions[0]
        if 1 < len(partial_positions) <= 8:
            choices = ", ".join(display_movie_title(position) for position in partial_positions)
            raise ValueError(f"Several movies match {query!r}: {choices}. Please be more specific.")
        if len(partial_positions) > 8:
            choices = ", ".join(display_movie_title(position) for position in partial_positions[:8])
            raise ValueError(f"Many movies match {query!r}; try a longer title. Some matches: {choices}")

    close_aliases = difflib.get_close_matches(
        normalized_query, list(title_alias_to_positions), n=5, cutoff=0.72
    )
    close_positions = sorted(
        {
            position
            for alias in close_aliases
            for position in title_alias_to_positions[alias]
        }
    )
    position_scores = {}
    for alias in close_aliases:
        alias_score = difflib.SequenceMatcher(None, normalized_query, alias).ratio()
        for position in title_alias_to_positions[alias]:
            position_scores[position] = max(position_scores.get(position, 0), alias_score)
    ranked_positions = sorted(position_scores, key=position_scores.get, reverse=True)
    if ranked_positions:
        best_position = ranked_positions[0]
        best_score = position_scores[best_position]
        second_score = position_scores[ranked_positions[1]] if len(ranked_positions) > 1 else 0
        if best_score >= 0.84 and best_score - second_score >= 0.06:
            return best_position
    if close_positions:
        choices = ", ".join(display_movie_title(position) for position in close_positions[:5])
        raise ValueError(f"Couldn't confidently resolve {query!r}. Did you mean: {choices}?")
    raise ValueError(f"No movie found for {query!r}. Check the spelling or try a longer title.")

print("Title search aliases are ready.")

Title search aliases are ready.


### Build the cosine-neighbor lookup

Fit a sparse nearest-neighbor index and define a helper that accepts a natural movie title, excludes the query movie itself, and returns the top 10 recommendations with their cosine similarity scores.

In [17]:
from sklearn.neighbors import NearestNeighbors

movie_neighbors = NearestNeighbors(metric="cosine", algorithm="brute", n_jobs=-1)
movie_neighbors.fit(combined_matrix)

def recommend_similar_movies(title, n_recommendations=10, feature_matrix=combined_matrix):
    """Return nearest catalog movies for a user-entered title, excluding the query."""
    if n_recommendations < 1:
        raise ValueError("n_recommendations must be at least 1.")

    query_position = resolve_movie_title(title)
    model = movie_neighbors if feature_matrix is combined_matrix else NearestNeighbors(
        metric="cosine", algorithm="brute", n_jobs=-1
    ).fit(feature_matrix)
    neighbor_count = min(n_recommendations + 1, len(movie_catalog))
    distances, indices = model.kneighbors(
        feature_matrix[query_position], n_neighbors=neighbor_count
    )

    recommendations = [
        (int(index), float(1 - distance))
        for index, distance in zip(indices[0], distances[0])
        if index != query_position
    ][:n_recommendations]
    result = movie_catalog.iloc[[index for index, _ in recommendations]][
        ["movieId", "title", "genres", "user_tag_count", "genome_score_count"]
    ].copy()
    result["title"] = [display_movie_title(index) for index, _ in recommendations]
    result["similarity"] = [score for _, score in recommendations]
    
    return result.reset_index(drop=True)

recommend_similar_movies("Toy Story (1995)", n_recommendations=10)

,movieId,title,genres,user_tag_count,genome_score_count,similarity
0,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,719,1128,0.741887
1,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,962,1128,0.682304
2,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,897,1128,0.678932
3,201588,Toy Story 4 (2019),Adventure|Animation|Children|Comedy,360,1128,0.667168
4,199694,Despicable Me Presents: Minion Madness (2010),Adventure|Animation|Children|Comedy,0,1128,0.665625
5,229773,Adventure Time: Islands (2017),Adventure|Animation|Comedy|Fantasy,0,1128,0.660758
6,56,Kids of the Round Table (1995),Adventure|Children|Comedy|Fantasy,0,1128,0.655605
7,2355,A Bug's Life (1998),Adventure|Animation|Children|Comedy,244,1128,0.647923
8,247988,Luca (2021),Adventure|Animation|Children|Comedy|Fantasy,191,1128,0.641955
9,2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,134,1128,0.641432


### Compare feature-group recommendations

Review the nearest neighbors for Toy Story using genres alone, user tags alone, Genome tags alone, and the combined representation. This is a qualitative sanity check; systematic rating-based evaluation belongs in the next milestone.

In [18]:
for feature_name, matrix in [
    ("Genres only", genre_matrix),
    ("User tags only", user_tag_matrix),
    ("Genome tags only", genome_matrix),
    ("Combined features", combined_matrix),
]:
    print(feature_name)
    display(recommend_similar_movies("Toy Story (1995)", n_recommendations=5, feature_matrix=matrix))

Genres only


,movieId,title,genres,user_tag_count,genome_score_count,similarity
0,286131,The Super Mario Bros. Movie (2023),Adventure|Animation|Children|Comedy|Fantasy,93,1128,1.0
1,260495,My Little Pony: A New Generation (2021),Adventure|Animation|Children|Comedy|Fantasy,3,0,1.0
2,213207,Onward (2020),Adventure|Animation|Children|Comedy|Fantasy,194,1128,1.0
3,117454,The Magic Crystal (2011),Adventure|Animation|Children|Comedy|Fantasy,4,0,1.0
4,200630,Missing Link (2019),Adventure|Animation|Children|Comedy|Fantasy,28,1128,1.0


User tags only


,movieId,title,genres,user_tag_count,genome_score_count,similarity
0,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,719,1128,0.279882
1,201588,Toy Story 4 (2019),Adventure|Animation|Children|Comedy,360,1128,0.195306
2,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,897,1128,0.188717
3,115879,Toy Story Toons: Small Fry (2011),Adventure|Animation|Children|Comedy|Fantasy,25,1128,0.125971
4,115875,Toy Story Toons: Hawaiian Vacation (2011),Adventure|Animation|Children|Comedy|Fantasy,17,1128,0.113373


Genome tags only


,movieId,title,genres,user_tag_count,genome_score_count,similarity
0,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,962,1128,0.946417
1,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,719,1128,0.945776
2,2355,A Bug's Life (1998),Adventure|Animation|Children|Comedy,244,1128,0.937993
3,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,897,1128,0.935209
4,6377,Finding Nemo (2003),Adventure|Animation|Children|Comedy,1051,1128,0.917575


Combined features


,movieId,title,genres,user_tag_count,genome_score_count,similarity
0,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,719,1128,0.741887
1,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,962,1128,0.682304
2,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,897,1128,0.678932
3,201588,Toy Story 4 (2019),Adventure|Animation|Children|Comedy,360,1128,0.667168
4,199694,Despicable Me Presents: Minion Madness (2010),Adventure|Animation|Children|Comedy,0,1128,0.665625


In [30]:
movie_query = input("Enter a movie name: ").strip()
print("Movies Similar to", movie_query.title())
display(recommend_similar_movies(movie_query, n_recommendations=10))

Movies Similar to The Matrix


,movieId,title,genres,user_tag_count,genome_score_count,similarity
0,1240,The Terminator (1984),Action|Sci-Fi|Thriller,1664,1128,0.681667
1,541,Blade Runner (1982),Action|Sci-Fi|Thriller,3929,1128,0.667908
2,5903,Equilibrium (2002),Action|Sci-Fi|Thriller,1251,1128,0.659495
3,6365,The Matrix Reloaded (2003),Action|Adventure|Sci-Fi|Thriller|IMAX,1236,1128,0.645372
4,203393,Escape Plan: The Extractors (2019),Action|Thriller,0,1128,0.631066
5,3527,Predator (1987),Action|Sci-Fi|Thriller,928,1128,0.630529
6,6934,The Matrix Revolutions (2003),Action|Adventure|Sci-Fi|Thriller|IMAX,1084,1128,0.629287
7,34319,The Island (2005),Action|Sci-Fi|Thriller,1066,1128,0.625825
8,2600,eXistenZ (1999),Action|Sci-Fi|Thriller,566,1128,0.625811
9,6597,"Princess Blade, The (Shura Yukihime) (2001)",Action|Sci-Fi,0,1128,0.624771


## Milestone 4: Offline evaluation and feature-weight tuning

Build a reproducible, per-user chronological train/validation/test split from ratings. Tune feature weights on validation users only, then evaluate the selected model once on a held-out test set and compare it with the existing genre-only recommender.

### Select evaluation users and load only their ratings

Scan the large ratings file in chunks to identify users with enough history, then load ratings for a deterministic sample. This keeps the evaluation repeatable without bringing the full ratings table into memory.

In [20]:
ratings_path = "../datasets/ratings.csv"
ratings_chunk_size = 1_000_000
minimum_user_ratings = 30
evaluation_sample_size = 1_200
evaluation_random_seed = 42

rating_counts_by_user = pd.Series(dtype="int64")
for ratings_chunk in pd.read_csv(
    ratings_path, usecols=["userId"], chunksize=ratings_chunk_size
):
    rating_counts_by_user = rating_counts_by_user.add(
        ratings_chunk["userId"].value_counts(), fill_value=0
    )

eligible_user_ids = rating_counts_by_user[
    rating_counts_by_user >= minimum_user_ratings
].index.to_numpy(dtype="int64")
rng = np.random.default_rng(evaluation_random_seed)
sampled_user_ids = rng.choice(
    eligible_user_ids,
    size=min(evaluation_sample_size, len(eligible_user_ids)),
    replace=False,
)
sampled_user_id_set = set(sampled_user_ids)

sampled_rating_chunks = []
for ratings_chunk in pd.read_csv(
    ratings_path,
    usecols=["userId", "movieId", "rating", "timestamp"],
    chunksize=ratings_chunk_size,
):
    sampled_chunk = ratings_chunk.loc[
        ratings_chunk["userId"].isin(sampled_user_id_set)
    ]
    if not sampled_chunk.empty:
        sampled_rating_chunks.append(sampled_chunk)

sampled_ratings = pd.concat(sampled_rating_chunks, ignore_index=True)
print(
    f"Eligible users: {len(eligible_user_ids):,}; sampled users: "
    f"{sampled_ratings['userId'].nunique():,}; sampled ratings: {len(sampled_ratings):,}"
)

Eligible users: 169,614; sampled users: 1,200; sampled ratings: 215,285


### Create chronological splits and relevant-item sets

For each sampled user, sort ratings by timestamp and split them into 70% train, 15% validation, and 15% test. A relevant recommendation is a held-out movie rated at least 4 stars. Retain only users with at least one positive movie in every split.

In [21]:
sampled_ratings = sampled_ratings.sort_values(
    ["userId", "timestamp", "movieId"], kind="stable"
)
evaluation_users = []

for user_id, user_ratings in sampled_ratings.groupby("userId", sort=False):
    user_ratings = user_ratings.drop_duplicates("movieId", keep="last")
    if len(user_ratings) < minimum_user_ratings:
        continue

    train_end = int(len(user_ratings) * 0.70)
    validation_end = int(len(user_ratings) * 0.85)
    train_ratings = user_ratings.iloc[:train_end]
    validation_ratings = user_ratings.iloc[train_end:validation_end]
    test_ratings = user_ratings.iloc[validation_end:]

    train_movie_ids = set(train_ratings["movieId"].astype("int64"))
    validation_movie_ids = set(validation_ratings["movieId"].astype("int64"))
    validation_relevant_ids = set(
        validation_ratings.loc[validation_ratings["rating"] >= 4.0, "movieId"].astype("int64")
    )
    test_relevant_ids = set(
        test_ratings.loc[test_ratings["rating"] >= 4.0, "movieId"].astype("int64")
    )
    train_positive_ids = set(
        train_ratings.loc[train_ratings["rating"] >= 4.0, "movieId"].astype("int64")
    )
    validation_positive_ids = validation_relevant_ids

    if train_positive_ids and validation_relevant_ids and test_relevant_ids:
        evaluation_users.append(
            {
                "user_id": int(user_id),
                "train_movie_ids": train_movie_ids,
                "validation_movie_ids": validation_movie_ids,
                "validation_positive_ids": validation_positive_ids,
                "train_positive_ids": train_positive_ids,
                "validation_profile_positive_ids": train_positive_ids,
                "test_profile_positive_ids": train_positive_ids | validation_positive_ids,
                "test_excluded_movie_ids": train_movie_ids | validation_movie_ids,
                "test_positive_ids": test_relevant_ids,
            }
        )

if len(evaluation_users) < 100:
    raise ValueError(
        "Fewer than 100 sampled users had positive train, validation, and test ratings. "
        "Increase evaluation_sample_size and rerun the sampling and split cells."
    )

rng.shuffle(evaluation_users)
evaluation_users = evaluation_users[:250]
print(f"Users retained for validation and test: {len(evaluation_users):,}")

Users retained for validation and test: 250


### Define ranking metrics and a reusable evaluator

Score the catalog from each user's positive-rating profile, exclude movies already rated before the evaluation period, and report macro Precision@10, Recall@10, NDCG@10, and catalog coverage.

In [24]:
from scipy.sparse import vstack

movie_id_to_row = {
    int(movie_id): row for row, movie_id in enumerate(catalog_movie_ids)
}
feature_blocks = {
    "genres": genre_matrix,
    "user_tags": user_tag_matrix,
    "genome": genome_matrix,
}
evaluation_k = 10

def make_weighted_item_matrix(weights):
    """Combine normalized feature blocks using weights and normalize item rows."""
    if set(weights) != set(feature_blocks):
        raise ValueError(f"Weights must define exactly {set(feature_blocks)}.")
    if any(weight < 0 for weight in weights.values()) or not np.isclose(sum(weights.values()), 1):
        raise ValueError("Feature weights must be non-negative and sum to 1.")
    blocks = [
        feature_blocks[name].multiply(np.sqrt(weights[name]))
        for name in ("genres", "user_tags", "genome")
    ]
    return normalize(hstack(blocks, format="csr", dtype=np.float32), norm="l2", copy=False).tocsr()

def evaluate_ranking(weights, split_name, user_records):
    """Evaluate held-out positives for a split without fitting on its labels."""
    item_matrix = make_weighted_item_matrix(weights)
    profile_field = (
        "validation_profile_positive_ids"
        if split_name == "validation"
        else "test_profile_positive_ids"
    )
    excluded_field = (
        "train_movie_ids" if split_name == "validation" else "test_excluded_movie_ids"
    )
    relevant_field = (
        "validation_positive_ids" if split_name == "validation" else "test_positive_ids"
    )

    profile_rows = []
    valid_user_records = []
    for record in user_records:
        profile_positions = [
            movie_id_to_row[movie_id]
            for movie_id in record[profile_field]
            if movie_id in movie_id_to_row
        ]
        relevant_positions = {
            movie_id_to_row[movie_id]
            for movie_id in record[relevant_field]
            if movie_id in movie_id_to_row
        }
        if not profile_positions or not relevant_positions:
            continue
        profile = item_matrix[profile_positions].mean(axis=0)
        profile = csr_matrix(np.asarray(profile, dtype=np.float32))
        profile = normalize(profile, norm="l2", copy=False).tocsr()
        profile_rows.append(profile)
        valid_user_records.append((record, relevant_positions))

    if not profile_rows:
        raise ValueError(f"No evaluable users for the {split_name} split.")

    user_profiles = vstack(profile_rows, format="csr", dtype=np.float32)
    all_scores = user_profiles @ item_matrix.T
    discounts = 1 / np.log2(np.arange(2, evaluation_k + 2))
    precisions, recalls, ndcgs = [], [], []
    recommended_movie_rows = set()

    for user_row, (record, relevant_positions) in enumerate(valid_user_records):
        scores = all_scores.getrow(user_row).toarray().ravel()
        excluded_ids = record[excluded_field]
        excluded_positions = [
            movie_id_to_row[movie_id]
            for movie_id in excluded_ids
            if movie_id in movie_id_to_row
        ]
        scores[excluded_positions] = -np.inf
        top_positions = np.argpartition(scores, -evaluation_k)[-evaluation_k:]
        top_positions = top_positions[np.argsort(scores[top_positions])[::-1]]
        top_list = top_positions.tolist()
        recommended_movie_rows.update(top_list)

        hits = np.array([position in relevant_positions for position in top_list], dtype=np.float32)
        precisions.append(float(hits.sum() / evaluation_k))
        recalls.append(float(hits.sum() / len(relevant_positions)))
        dcg = float(np.sum(hits * discounts))
        ideal_hits = min(len(relevant_positions), evaluation_k)
        idcg = float(discounts[:ideal_hits].sum())
        ndcgs.append(dcg / idcg if idcg else 0.0)

    return {
        "users": len(valid_user_records),
        "precision@10": float(np.mean(precisions)),
        "recall@10": float(np.mean(recalls)),
        "ndcg@10": float(np.mean(ndcgs)),
        "catalog_coverage": len(recommended_movie_rows) / len(movie_catalog),
    }

print(f"Ranking metrics ready for top-{evaluation_k} evaluation.")

Ranking metrics ready for top-10 evaluation.


### Tune feature weights on validation only

Compare a small, predefined weight grid against the genre-only baseline using validation NDCG@10. Do not use test ratings to pick weights; the held-out test set is reserved for the next cell.

In [25]:
weight_candidates = [
    {"genres": 1 / 3, "user_tags": 1 / 3, "genome": 1 / 3},
    {"genres": 0.60, "user_tags": 0.20, "genome": 0.20},
    {"genres": 0.20, "user_tags": 0.60, "genome": 0.20},
    {"genres": 0.20, "user_tags": 0.20, "genome": 0.60},
    {"genres": 0.50, "user_tags": 0.25, "genome": 0.25},
    {"genres": 0.20, "user_tags": 0.40, "genome": 0.40},
]

validation_results = []
for candidate in weight_candidates:
    metrics = evaluate_ranking(candidate, "validation", evaluation_users)
    validation_results.append({**candidate, **metrics, "model": "combined"})

genre_baseline_weights = {"genres": 1.0, "user_tags": 0.0, "genome": 0.0}
genre_validation_metrics = evaluate_ranking(
    genre_baseline_weights, "validation", evaluation_users
)
validation_results.append(
    {**genre_baseline_weights, **genre_validation_metrics, "model": "genre-only baseline"}
)

validation_results_df = pd.DataFrame(validation_results)
combined_validation_results = validation_results_df.loc[
    validation_results_df["model"].eq("combined")
].sort_values(["ndcg@10", "recall@10"], ascending=False)
if combined_validation_results.empty:
    raise ValueError("Validation produced no combined-model results.")
selected_weights = {
    "genres": float(combined_validation_results.iloc[0]["genres"]),
    "user_tags": float(combined_validation_results.iloc[0]["user_tags"]),
    "genome": float(combined_validation_results.iloc[0]["genome"]),
}
display(
    validation_results_df[
        ["model", "genres", "user_tags", "genome", "users", "precision@10", "recall@10", "ndcg@10", "catalog_coverage"]
    ].sort_values("ndcg@10", ascending=False).reset_index(drop=True)
)
print(f"Selected weights from validation NDCG@10: {selected_weights}")

,model,genres,user_tags,genome,users,precision@10,recall@10,ndcg@10,catalog_coverage
0,combined,0.600000,0.200000,0.200000,250,0.0068,0.007357,0.008043,0.005154
1,combined,0.500000,0.250000,0.250000,250,0.0044,0.004795,0.005474,0.004264
2,genre-only baseline,1.000000,0.000000,0.000000,250,0.0024,0.003205,0.003004,0.012561
3,combined,0.200000,0.200000,0.600000,250,0.0028,0.002215,0.002800,0.003178
4,combined,0.333333,0.333333,0.333333,250,0.0024,0.003173,0.002722,0.002600
5,combined,0.200000,0.400000,0.400000,250,0.0004,0.000500,0.000337,0.001572
6,combined,0.200000,0.600000,0.200000,250,0.0000,0.000000,0.000000,0.001421


Selected weights from validation NDCG@10: {'genres': 0.6, 'user_tags': 0.2, 'genome': 0.2}


### Evaluate once on the held-out test set

Compare the selected combined weights with the genre-only baseline on test positives. Test outcomes are for reporting only and must not be used to retune the weights.

In [26]:
test_results = []
for model_name, weights in [
    ("Selected combined model", selected_weights),
    ("Genre-only baseline", genre_baseline_weights),
]:
    metrics = evaluate_ranking(weights, "test", evaluation_users)
    test_results.append({"model": model_name, **weights, **metrics})

test_results_df = pd.DataFrame(test_results)
display(test_results_df)

selected_test_ndcg = test_results_df.loc[
    test_results_df["model"].eq("Selected combined model"), "ndcg@10"
].iloc[0]
baseline_test_ndcg = test_results_df.loc[
    test_results_df["model"].eq("Genre-only baseline"), "ndcg@10"
].iloc[0]
ndcg_change = selected_test_ndcg - baseline_test_ndcg
print(f"Test NDCG@10 change vs genre-only baseline: {ndcg_change:+.4f}")

,model,genres,user_tags,genome,users,precision@10,recall@10,ndcg@10,catalog_coverage
0,Selected combined model,0.6,0.2,0.2,250,0.0036,0.004035,0.003548,0.004923
1,Genre-only baseline,1.0,0.0,0.0,250,0.0012,0.001602,0.001388,0.011972


Test NDCG@10 change vs genre-only baseline: +0.0022
